# Temporal Crosscoders — Toy Model

We test whether a **temporal crosscoder** — an SAE that sees a window of $T$ consecutive activation vectors rather than a single one — can better recover true underlying features when those features exhibit **temporal correlation**.

The ground-truth setup (50 orthogonal features in $\mathbb{R}^{100}$) mirrors [Chanin et al. (2025)](https://arxiv.org/abs/2408.05147) and the original `sae_token_embeddings` notebook.

## Architecture comparison

| Model | Input | Encoder | Decoder |
|-------|-------|---------|----------|
| **Standard SAE** | $x_t \in \mathbb{R}^d$ | $W_\text{enc} \in \mathbb{R}^{h \times d}$ | $W_\text{dec} \in \mathbb{R}^{d \times h}$ |
| **Temporal Crosscoder** | $[x_t; \ldots; x_{t+T-1}] \in \mathbb{R}^{Td}$ | $W_\text{enc} \in \mathbb{R}^{h \times Td}$ | $W_\text{dec}^{(s)} \in \mathbb{R}^{d \times h}$ per position $s$ |

The crosscoder shares one latent code across all positions in the window, but reconstructs each position separately via its own decoder slice.

## Experiments

| Exp | Data generating process | Expected crosscoder advantage |
|-----|------------------------|-------------------------------|
| 1 | i.i.d. (no temporal correlation) | None — crosscoder ≈ SAE |
| 2 | **Scheme A** — AR(1) magnitude correlation, $\rho=0.9$ | Weak — attenuated by sparsity |
| 3 | **Scheme B** — Gaussian copula support, persistent | Moderate |
| 4 | **Scheme C** — 2-state Markov chain support, $\alpha=0.95, \beta=0.03$ | Strong |


## 0. Install dependencies

In [25]:
!pip install sae-lens -q

## 1. Imports & shared helpers

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Any, Callable, NamedTuple
from tqdm.auto import tqdm
from scipy.stats import norm
from torch.distributions import MultivariateNormal
import random
import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from transformer_lens.hook_points import HookedRootModule
from sae_lens.training.sae_trainer import SAETrainer
from sae_lens.config import SAETrainerConfig, LoggingConfig
from sae_lens import TrainingSAE, StandardTrainingSAE, StandardTrainingSAEConfig

DEFAULT_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print(f'Device: {DEFAULT_DEVICE}')

Device: cpu


## 2. Toy model — 50 orthogonal features in $\mathbb{R}^{100}$

In [27]:
# ──────────────────────────────────────────────
# Toy model (same as sae_token_embeddings.ipynb)
# ──────────────────────────────────────────────

def orthogonalize(num_vectors: int, vector_len: int, target_cos_sim: float = 0) -> torch.Tensor:
    """Optimise embeddings so all pairwise cosine similarities ≈ target_cos_sim."""
    embeddings = torch.randn(num_vectors, vector_len)
    embeddings = embeddings / embeddings.norm(p=2, dim=1, keepdim=True)
    embeddings.requires_grad_(True)
    optimizer = torch.optim.Adam([embeddings], lr=0.01)
    pbar = tqdm(range(1000), desc='Orthogonalising features', leave=False)
    for _ in pbar:
        optimizer.zero_grad()
        dot = embeddings @ embeddings.T
        diff = dot - target_cos_sim
        diff.fill_diagonal_(0)
        loss = diff.pow(2).sum() + num_vectors * (dot.diag() - 1).pow(2).sum()
        loss.backward()
        optimizer.step()
        pbar.set_description(f'Orthogonalising | loss={loss.item():.3f}')
    with torch.no_grad():
        embeddings = embeddings / embeddings.norm(p=2, dim=1, keepdim=True)
    return embeddings.detach().clone()


class ToyModel(HookedRootModule):
    """Linear map from feature space to representation space."""
    def __init__(self, num_feats: int, hidden_dim: int, target_cos_sim: float = 0):
        super().__init__()
        self.embed = nn.Linear(num_feats, hidden_dim, bias=False)
        embeddings = orthogonalize(num_feats, hidden_dim, target_cos_sim)
        self.embed.weight.data = embeddings.T   # (hidden_dim, num_feats)
        self.setup()

    def forward(self, x: torch.Tensor, **kwargs: Any):
        return self.embed(x)


NUM_FEATS  = 50
HIDDEN_DIM = 100
FEAT_PROB  = 0.05   # sparse: ~2.5 features fire per position
FEAT_MEAN  = 1.0
FEAT_STD   = 0.15

toy_model = ToyModel(NUM_FEATS, HIDDEN_DIM).to(DEFAULT_DEVICE)
toy_model.eval()

feat_probs = torch.ones(NUM_FEATS) * FEAT_PROB
print(f'Toy model: {NUM_FEATS} features | d={HIDDEN_DIM} | p={FEAT_PROB}')

Orthogonalising features:   0%|          | 0/1000 [00:00<?, ?it/s]

Toy model: 50 features | d=100 | p=0.05


## 3. Temporal data generators

Each generator produces sequences of shape `(num_seqs, T, num_feats)` — **before** passing through the toy model. The four schemes follow directly from the problem statement:

- **IID** (Exp 1): $s_{i,t} \sim \text{Bernoulli}(p)$ and $m_{i,t} \sim |\mathcal{N}(\mu, \sigma^2)|$, both independent.
- **Scheme A** (Exp 2): AR(1) magnitude process; support still i.i.d.
- **Scheme B** (Exp 3): Gaussian copula over support indicators; magnitudes i.i.d.
- **Scheme C** (Exp 4): Two-state Markov chain support; magnitudes i.i.d.

In [28]:
# ──────────────────────────────────────────────────────────────────────────
# IID baseline
# ──────────────────────────────────────────────────────────────────────────

def generate_iid_sequences(
    num_seqs: int,
    T: int,
    p: float = FEAT_PROB,
    mu: float = FEAT_MEAN,
    sigma: float = FEAT_STD,
    num_feats: int = NUM_FEATS,
    device: torch.device = DEFAULT_DEVICE,
) -> torch.Tensor:
    """Returns (num_seqs, T, num_feats) — all i.i.d. Bernoulli gates × |N(mu,sigma)| magnitudes."""
    s = torch.bernoulli(torch.full((num_seqs, T, num_feats), p, device=device))
    m = (torch.randn(num_seqs, T, num_feats, device=device) * sigma + mu).abs()
    return s * m


# ──────────────────────────────────────────────────────────────────────────
# Scheme A — AR(1) magnitude correlation
#
# m_{i,t} = mu + rho*(m_{i,t-1} - mu) + eps_{i,t},  eps ~ N(0, sigma^2*(1-rho^2))
# Support still i.i.d.
# ──────────────────────────────────────────────────────────────────────────

def generate_scheme_a_sequences(
    num_seqs: int,
    T: int,
    rho: float = 0.9,          # AR(1) autocorrelation
    p: float = FEAT_PROB,
    mu: float = FEAT_MEAN,
    sigma: float = FEAT_STD,
    num_feats: int = NUM_FEATS,
    device: torch.device = DEFAULT_DEVICE,
) -> torch.Tensor:
    """Scheme A: temporally correlated magnitudes via AR(1).  (num_seqs, T, num_feats)"""
    # Innovation std ensures stationary marginal variance = sigma^2
    innov_std = sigma * np.sqrt(1 - rho**2)

    # Initialise from stationary distribution
    m = torch.randn(num_seqs, num_feats, device=device) * sigma + mu   # (N, F)
    m_list = [m]
    for _ in range(T - 1):
        eps = torch.randn_like(m) * innov_std
        m = mu + rho * (m - mu) + eps
        m_list.append(m)

    magnitudes = torch.stack(m_list, dim=1).abs()   # (N, T, F)
    support    = torch.bernoulli(torch.full((num_seqs, T, num_feats), p, device=device))
    return support * magnitudes


# ──────────────────────────────────────────────────────────────────────────
# Scheme B — Gaussian copula over support
#
# Latent z_{i,t} are jointly Gaussian with AR(1) covariance;
# s_{i,t} = 1[z_{i,t} > Phi^{-1}(1-p)]   =>  E[s] = p for all t.
# ──────────────────────────────────────────────────────────────────────────

def _ar1_covariance(T: int, rho: float) -> torch.Tensor:
    """T×T AR(1) correlation matrix."""
    idx = torch.arange(T).float()
    return rho ** torch.abs(idx.unsqueeze(0) - idx.unsqueeze(1))


def generate_scheme_b_sequences(
    num_seqs: int,
    T: int,
    rho: float = 0.85,         # support autocorrelation
    p: float = FEAT_PROB,
    mu: float = FEAT_MEAN,
    sigma: float = FEAT_STD,
    num_feats: int = NUM_FEATS,
    device: torch.device = DEFAULT_DEVICE,
) -> torch.Tensor:
    """Scheme B: Gaussian copula support.  (num_seqs, T, num_feats)"""
    cov = _ar1_covariance(T, rho).to(device)
    mvn = MultivariateNormal(torch.zeros(T, device=device), covariance_matrix=cov)

    threshold = float(norm.ppf(1 - p))   # Phi^{-1}(1-p)

    # Sample latent: (num_seqs, num_feats, T)  then transpose
    z = mvn.sample((num_seqs, num_feats))          # (N, F, T)
    z = z.permute(0, 2, 1)                          # (N, T, F)
    support = (z > threshold).float()

    magnitudes = (torch.randn(num_seqs, T, num_feats, device=device) * sigma + mu).abs()
    return support * magnitudes


# ──────────────────────────────────────────────────────────────────────────
# Scheme C — 2-state Markov chain support
#
# P(s_t=1 | s_{t-1}=1) = alpha   (stay-on probability)
# P(s_t=1 | s_{t-1}=0) = beta    (turn-on probability)
# Stationary p = beta / (1 - alpha + beta)
# ──────────────────────────────────────────────────────────────────────────

def generate_scheme_c_sequences(
    num_seqs: int,
    T: int,
    alpha: float = 0.95,       # stay-on
    beta:  float = 0.03,       # turn-on
    mu: float = FEAT_MEAN,
    sigma: float = FEAT_STD,
    num_feats: int = NUM_FEATS,
    device: torch.device = DEFAULT_DEVICE,
) -> torch.Tensor:
    """Scheme C: 2-state Markov chain support.  (num_seqs, T, num_feats)"""
    p_stat = beta / (1 - alpha + beta)     # stationary probability

    # Initialise from stationary distribution
    s = torch.bernoulli(torch.full((num_seqs, num_feats), p_stat, device=device))
    support_list = [s.clone()]

    for _ in range(T - 1):
        # Transition probabilities for each cell
        p_next = torch.where(s == 1,
                             torch.full_like(s, alpha),
                             torch.full_like(s, beta))
        s = torch.bernoulli(p_next)
        support_list.append(s.clone())

    support    = torch.stack(support_list, dim=1)                    # (N, T, F)
    magnitudes = (torch.randn(num_seqs, T, num_feats, device=device) * sigma + mu).abs()
    return support * magnitudes


# ──────────────────────────────────────────────────────────────────────────
# Diagnostic: measure empirical autocorrelation at lag 1
# ──────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def empirical_lag1_autocorr(
    feat_seqs: torch.Tensor,   # (N, T, F)
) -> float:
    """Mean lag-1 autocorrelation of activation magnitudes across features."""
    a = feat_seqs            # (N, T, F)
    a_t  = a[:, :-1, :]     # (N, T-1, F)
    a_t1 = a[:,  1:, :]     # (N, T-1, F)

    # Per-feature autocorrelation: average over (N, T-1)
    def corr(x, y):
        x = x - x.mean()
        y = y - y.mean()
        return (x * y).sum() / (x.norm() * y.norm() + 1e-8)

    corrs = []
    for f in range(feat_seqs.shape[-1]):
        xf = a_t[:, :, f].flatten()
        yf = a_t1[:, :, f].flatten()
        corrs.append(corr(xf, yf).item())
    return float(np.mean(corrs))


# Quick sanity check
T_SEQ = 8   # window length used throughout
N_DIAG = 2000

for name, seqs in [
    ('IID',      generate_iid_sequences(N_DIAG, T_SEQ)),
    ('Scheme A', generate_scheme_a_sequences(N_DIAG, T_SEQ, rho=0.9)),
    ('Scheme B', generate_scheme_b_sequences(N_DIAG, T_SEQ, rho=0.85)),
    ('Scheme C', generate_scheme_c_sequences(N_DIAG, T_SEQ, alpha=0.95, beta=0.03)),
]:
    ac = empirical_lag1_autocorr(seqs)
    print(f'{name:12s}  lag-1 autocorr = {ac:.4f}')

IID           lag-1 autocorr = -0.0016
Scheme A      lag-1 autocorr = 0.0002
Scheme B      lag-1 autocorr = 0.5202
Scheme C      lag-1 autocorr = 0.8878


## 4. Temporal Crosscoder architecture (TopK)

The crosscoder takes a window of $T$ consecutive activations and produces a single latent code $u$ shared across the window. Each position gets its own decoder slice.

$$u = \\text{TopK}_k\\bigl(W_\\text{enc}[x_t; \\ldots; x_{t+T-1}] + b_\\text{enc}\\bigr)$$
$$\\hat{x}_{t+s} = W_\\text{dec}^{(s)}\\, u + b_\\text{dec}^{(s)}, \\quad s = 0, \\ldots, T-1$$
$$\\mathcal{L} = \\sum_{s=0}^{T-1} \\|\\hat{x}_{t+s} - x_{t+s}\\|_2^2$$

**TopK replaces ReLU + L1** — exactly $k=11$ latents fire per window. This eliminates the L1 coefficient hyperparameter that caused all previous failures.

$k=11$ rationale: under IID with $p=0.05$, $T=8$, the expected number of unique features appearing at least once is $d_{\\text{sae}}(1-(1-p)^T) \\approx 17$. $k=11$ is slightly below this to keep codes crisp.

In [29]:
# TopK crosscoder: exactly k latents fire per window — no L1 tuning needed.
#
# k=11 rationale:
#   E[features active at ≥1 position in window] = d_sae*(1-(1-p)^T) = 50*0.34 ≈ 17
#   k=11 is slightly below the expected natural sparsity, keeping representations
#   crisp without over-forcing sparsity (which would kill weaker/rarer features).
#   For IID: a single token has ~2.5 active features; across T=8 positions we
#   expect ~17 unique features to appear at least once → k=11 is in range.
#
# TopK vs ReLU+L1:
#   - Eliminates the L1 coefficient as a hyperparameter (was the main failure mode)
#   - Gradient flows through exactly k paths every step → stable training
#   - Dead latent problem is dramatically reduced (all latents compete to be top-k)

TOP_K = 17   # latents active per window

class TemporalCrosscoder(nn.Module):
    """
    Temporal crosscoder with TopK activation.
    Encoder:  W_enc ∈ R^{h × (T*d_in)}
    Decoders: W_dec ∈ R^{T × d_in × h}  — one decoder slice per position
    Activation: TopK — exactly TOP_K latents active per forward pass
    """
    def __init__(self, d_in: int, d_sae: int, T: int, k: int = TOP_K):
        super().__init__()
        self.d_in  = d_in
        self.d_sae = d_sae
        self.T     = T
        self.k     = k

        self.b_dec = nn.Parameter(torch.zeros(T, d_in))          # per-position bias
        self.W_enc = nn.Parameter(torch.empty(d_sae, T * d_in))  # (h, T*d)
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.empty(T, d_in, d_sae))   # (T, d, h)

        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        with torch.no_grad():
            self._normalize_decoder()

    @torch.no_grad()
    def _normalize_decoder(self):
        norms = self.W_dec.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self.W_dec.data = self.W_dec.data / norms

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d) → u: (B, h)  with exactly k non-zero entries per row."""
        x_centered = x - self.b_dec.unsqueeze(0)          # (B, T, d)
        x_flat     = x_centered.reshape(x.shape[0], -1)   # (B, T*d)
        pre        = x_flat @ self.W_enc.T + self.b_enc   # (B, h)  pre-activations

        # TopK: zero out all but the k largest pre-activations (per sample)
        topk_vals, topk_idx = pre.topk(self.k, dim=-1)    # (B, k)
        u = torch.zeros_like(pre)
        u.scatter_(-1, topk_idx, F.relu(topk_vals))       # only keep positive top-k
        return u                                           # (B, h)

    def decode(self, u: torch.Tensor) -> torch.Tensor:
        """u: (B, h) → x_hat: (B, T, d)"""
        return torch.einsum('tdh,bh->btd', self.W_dec, u) + self.b_dec.unsqueeze(0)

    def forward(self, x: torch.Tensor):
        """x: (B, T, d) → (recon_loss, x_hat, u)  — no L1 term needed with TopK"""
        u          = self.encode(x)
        x_hat      = self.decode(u)
        recon_loss = (x_hat - x).pow(2).sum(dim=-1).mean()   # mean over (B, T)
        return recon_loss, x_hat, u

    @torch.no_grad()
    def decoder_directions(self, pos: int = 0) -> torch.Tensor:
        """(d, h) decoder columns for position pos."""
        return self.W_dec[pos]


## 5. Training & evaluation helpers

In [30]:
# ─── WHY DOES THE SHUFFLED SAE ALWAYS GET ~1.0? ──────────────────────────────
# The original DataIterator calls batch_fn(batch_size) which randomly picks ONE
# timestep per sequence.  Since all our temporal processes are *stationary*, the
# marginal distribution at any single position is the same Bernoulli × |N(μ,σ)|
# regardless of scheme.  The SAE therefore sees i.i.d. samples from the same
# marginal in every experiment → of course it always recovers features perfectly.
#
# The temporal correlation is completely invisible to a shuffled per-token model.
#
# ─── FIX: TWO SAE TRAINING MODES ─────────────────────────────────────────────
#
# SHUFFLED (baseline):  randomly sample one position per sequence per batch.
#   → SAE sees pure marginal; temporal structure is destroyed.
#   → Expected: near-perfect recovery for all schemes.
#
# SEQUENTIAL (new):     emit T consecutive tokens from each sequence IN ORDER.
#   → Consecutive tokens within a mini-batch are temporally correlated.
#   → Effectively shrinks the independent batch size by the autocorrelation time.
#   → For Scheme C (ρ≈0.84), an 'effective batch size' ≈ batch/(1+2ρ/(1-ρ)) ≈ batch/12
#   → Expected: still converges (same optimal solution), but slower / noisier
#     for highly autocorrelated schemes, potentially missing rarer features.

class ShuffledDataIterator:
    """
    Original behaviour: randomly sample ONE position from each sequence per call.
    The SAE sees samples from the stationary marginal, independent of temporal scheme.
    """
    def __init__(self, model: ToyModel, seq_gen_fn: Callable, batch_size: int, T: int):
        self.model      = model
        self.seq_gen_fn = seq_gen_fn
        self.batch_size = batch_size
        self.T          = T

    @torch.no_grad()
    def next_batch(self) -> torch.Tensor:
        # Generate batch_size sequences, pick ONE random timestep from each
        feat_seqs = self.seq_gen_fn(self.batch_size, self.T)          # (B, T, k)
        t_idx = torch.randint(0, self.T, (self.batch_size,))          # random per seq
        feats = feat_seqs[torch.arange(self.batch_size), t_idx]       # (B, k)
        return self.model(feats)                                       # (B, d)

    def __iter__(self):  return self
    def __next__(self):  return self.next_batch()


class SequentialDataIterator:
    """
    New: emit all T positions from each sequence IN TEMPORAL ORDER.
    Consecutive tokens in a mini-batch are correlated — the SAE gradient
    now carries autocorrelation structure from the generating process.

    For IID:      no difference from shuffled.
    For Scheme C: consecutive batch elements are highly correlated (same
                  features active), effectively reducing independent batch size.
    """
    def __init__(self, model: ToyModel, seq_gen_fn: Callable, batch_size: int, T: int):
        self.model      = model
        self.seq_gen_fn = seq_gen_fn
        self.batch_size = batch_size
        self.T          = T

    @torch.no_grad()
    def next_batch(self) -> torch.Tensor:
        # Generate ceil(B/T) sequences, flatten positions in order: t0,t1,...,tT-1,t0,t1,...
        n_seq = max(1, self.batch_size // self.T)
        feat_seqs = self.seq_gen_fn(n_seq, self.T)       # (n_seq, T, k)
        act_seqs  = self.model(feat_seqs)                # (n_seq, T, d)
        flat      = act_seqs.reshape(-1, HIDDEN_DIM)     # (n_seq*T, d)  ← in temporal order
        return flat[:self.batch_size]                    # (B, d)

    def __iter__(self):  return self
    def __next__(self):  return self.next_batch()


def _save_checkpoint_null(trainer, checkpoint_name, wandb_aliases=None): pass


def train_standard_sae(
    seq_gen_fn: Callable,          # (N, T) → (N, T, num_feats)  feature sequences
    toy_model: ToyModel,
    mode: str = 'shuffled',        # 'shuffled' | 'sequential'
    d_sae: int = NUM_FEATS,
    l1_coefficient: float = 1.0,
    l1_warm_up_steps: int = 5_000,
    training_tokens: int = 5_000_000,
    batch_size: int = 1024,
    device: torch.device = DEFAULT_DEVICE,
) -> StandardTrainingSAE:
    """Train an SAE with either shuffled (marginal) or sequential (correlated) token stream."""
    assert mode in ('shuffled', 'sequential'), f'Unknown mode: {mode}'
    tqdm._instances.clear()   # type: ignore
    cfg = StandardTrainingSAEConfig(
        l1_coefficient=l1_coefficient,
        l1_warm_up_steps=l1_warm_up_steps,
        d_in=toy_model.embed.weight.shape[0],
        d_sae=d_sae,
    )
    sae = StandardTrainingSAE(cfg)

    training_cfg = SAETrainerConfig(
        n_checkpoints=0,
        checkpoint_path='/tmp/sae_ckpt',
        total_training_samples=training_tokens,
        device=str(device),
        autocast=False,
        lr=3e-4,
        lr_end=3e-4,
        lr_scheduler_name='constant',
        lr_warm_up_steps=0,
        adam_beta1=0.9,
        adam_beta2=0.999,
        lr_decay_steps=0,
        n_restart_cycles=1,
        train_batch_size_samples=batch_size,
        dead_feature_window=1000,
        feature_sampling_window=2000,
        logger=LoggingConfig(log_to_wandb=False),
    )
    toy_model.eval()
    IterCls  = ShuffledDataIterator if mode == 'shuffled' else SequentialDataIterator
    iterator = IterCls(toy_model, seq_gen_fn, batch_size, T=T_SEQ)
    trainer  = SAETrainer(data_provider=iterator, sae=sae, cfg=training_cfg)
    trainer.fit()
    return sae


# ─── Temporal crosscoder training ──────────────────────────────────────────
#
# TOKEN-BUDGET MATCHING
# ─────────────────────
# The SAE trains for `training_tokens` individual tokens.
# The crosscoder trains on windows of length T, so a fair comparison gives it
# `training_tokens // T` windows → same number of token-level observations.
#
#   SAE:         5_000_000 tokens ÷ 1024 batch  = ~4,883 gradient steps
#   Crosscoder:  5_000_000 ÷ T=8 = 625,000 windows ÷ 512 batch = ~1,220 steps
#
# The old code used n_steps=20_000 × 512 batch × T=8 = 81.9M token-equivalents
# — 16× more data than the SAE. All comparisons were unfair.

TRAINING_TOKENS = 5_000_000   # SAE token budget
SAE_BATCH       = 1024
CC_BATCH        = 512

# ── Budget rationale ──────────────────────────────────────────────────────────
# "Fair" comparison for crosscoder vs SAE is genuinely ambiguous:
#
#   Token-budget match:  crosscoder gets training_tokens // T windows
#     → same raw data exposure, but only ~1,220 gradient steps vs SAE's ~4,883
#     → optimizer convergence is step-count dependent; crosscoder is undertrained
#
#   Step-budget match:   crosscoder gets same number of gradient steps as SAE
#     → same optimization budget; crosscoder naturally sees T× more token-context
#       per step (its architectural feature window), which is NOT a cheat —
#       it's exactly the information advantage we're trying to measure
#
# We use STEP-BUDGET matching.  Both models run for SAE_STEPS gradient steps.
# Token exposure is reported transparently in the summary table.
SAE_STEPS = TRAINING_TOKENS // SAE_BATCH   # ~4,883 steps

# ── Crosscoder step budget ────────────────────────────────────────────────────
# The crosscoder encoder has T×d_in = 800 input dimensions vs SAE's d_in = 100.
# With 8× more parameters in the encoder, each gradient step covers less of the
# loss landscape per parameter. To give the crosscoder a fair convergence
# opportunity we scale its step budget by T:
#
#   CC_STEPS = SAE_STEPS * T  ≈ 4,883 × 8 = 39,062 steps
#
# Each step uses batch=512 windows of T=8 tokens → total token-equivalents:
#   39,062 × 512 × 8 = 160M  (vs SAE's 5M — reported in summary for transparency)
#
# This is a deliberate asymmetry: we are asking "given unlimited training time,
# does the crosscoder architecture extract more information from temporal data?"
# not "do they converge at the same wall-clock speed?"
CC_STEPS = SAE_STEPS * T_SEQ   # ~39,062 steps

# No L1 coefficient needed — TopK enforces exact sparsity (k=TOP_K active per window).
# This eliminates the main hyperparameter failure mode from all previous versions.


def train_crosscoder(
    seq_gen_fn: Callable,
    toy_model: ToyModel,
    T: int = T_SEQ,
    d_sae: int = NUM_FEATS,
    k: int = TOP_K,              # TopK: exactly k latents active per window
    n_steps: int = CC_STEPS,     # T× SAE steps to compensate for T× wider encoder
    batch_size: int = CC_BATCH,
    lr: float = 3e-4,
    device: torch.device = DEFAULT_DEVICE,
) -> TemporalCrosscoder:
    """
    Train a TopK temporal crosscoder.

    TopK (k=11) replaces ReLU + L1:
      - Exactly k latents active per window — no sparsity hyperparameter
      - Gradient flows through k paths every step → stable, no dead latent problem
      - k=11 chosen slightly below the ~17 expected under IID+T=8 to ensure crisp codes

    Step budget = CC_STEPS = SAE_STEPS × T so the crosscoder gets proportionally
    more gradient updates to account for its T× wider encoder input.
    """
    tokens_seen = n_steps * batch_size * T
    print(f'  Crosscoder (TopK k={k}): {n_steps:,} steps × {batch_size} batch × T={T}'
          f' = {tokens_seen:,} token-equivalents'
          f'  (SAE: {SAE_STEPS:,} steps × {SAE_BATCH} = {TRAINING_TOKENS:,} tokens)')

    crosscoder = TemporalCrosscoder(d_in=HIDDEN_DIM, d_sae=d_sae, T=T, k=k).to(device)
    optimizer  = torch.optim.Adam(crosscoder.parameters(), lr=lr, betas=(0.9, 0.999))

    toy_model.eval()
    pbar = tqdm(range(n_steps), desc='Training crosscoder')

    for step in pbar:
        feat_seqs = seq_gen_fn(batch_size, T).to(device)
        with torch.no_grad():
            act_seqs = toy_model(feat_seqs)                # (B, T, d)

        loss, _, u = crosscoder(act_seqs)                  # no l1_coeff — TopK handles sparsity
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(crosscoder.parameters(), 1.0)
        optimizer.step()
        crosscoder._normalize_decoder()

        if step % max(1, n_steps // 20) == 0:
            n_active = (u > 0).float().sum(dim=-1).mean().item()   # should be ≈ k
            pbar.set_postfix(loss=f'{loss.item():.4f}', active=f'{n_active:.1f}/{d_sae}')

    return crosscoder


# ─── Feature recovery metrics ──────────────────────────────────────────────

def cos_sims(mat1: torch.Tensor, mat2: torch.Tensor) -> torch.Tensor:
    """Cosine similarities between columns of mat1 and mat2. Returns (h1, h2)."""
    n1 = mat1 / mat1.norm(dim=0, keepdim=True).clamp(min=1e-8)
    n2 = mat2 / mat2.norm(dim=0, keepdim=True).clamp(min=1e-8)
    return n1.T @ n2


@torch.no_grad()
def feature_recovery_score(
    decoder_directions: torch.Tensor,    # (d, h) — decoder columns
    true_features: torch.Tensor,         # (d, k) — true feature columns
) -> dict:
    """
    Compute feature recovery metrics:
      - mean_max_cos_sim:  average over true features of max |cos sim| with any decoder col
      - frac_recovered:    fraction of true features with max |cos sim| >= 0.9
    """
    sims = cos_sims(decoder_directions, true_features).abs()   # (h, k)
    max_per_true = sims.max(dim=0).values                       # (k,)
    return {
        'mean_max_cos_sim': max_per_true.mean().item(),
        'frac_recovered_90': (max_per_true >= 0.9).float().mean().item(),
        'frac_recovered_80': (max_per_true >= 0.8).float().mean().item(),
        'per_feature': max_per_true.cpu().numpy(),
    }


def print_recovery(name: str, metrics: dict):
    print(f'[{name}]  mean-max cos-sim={metrics["mean_max_cos_sim"]:.3f}  '
          f'| recovered@0.9={metrics["frac_recovered_90"]:.1%}  '
          f'| recovered@0.8={metrics["frac_recovered_80"]:.1%}')


# ─── Plotting ──────────────────────────────────────────────────────────────

def plot_recovery_heatmap(
    decoder_directions: torch.Tensor,    # (d, h)
    true_features: torch.Tensor,         # (d, k)
    title: str,
    height: int = 420,
    width: int = 520,
):
    sims = cos_sims(decoder_directions, true_features)   # (h, k)
    fig = go.Figure(go.Heatmap(
        z=sims.detach().cpu().numpy(),
        zmin=-1, zmax=1,
        colorscale='RdBu',
        colorbar=dict(title='cos sim', dtick=1, tickvals=[-1, 0, 1]),
        hovertemplate='True feature: %{x}<br>SAE latent: %{y}<br>Cos sim: %{z:.3f}<extra></extra>',
    ))
    fig.update_layout(
        title=title,
        xaxis_title='True feature',
        yaxis_title='SAE / crosscoder latent',
        height=height, width=width,
    )
    fig.show()


def plot_summary_bar(
    results: list[dict],
    metric: str = 'mean_max_cos_sim',
):
    """Bar chart comparing feature recovery across experiments."""
    labels  = [r['label']     for r in results]
    sae_v   = [r['sae'][metric]   for r in results]
    cross_v = [r['crosscoder'][metric] for r in results]

    fig = go.Figure([
        go.Bar(name='Standard SAE',        x=labels, y=sae_v,   marker_color='steelblue'),
        go.Bar(name='Temporal Crosscoder', x=labels, y=cross_v, marker_color='firebrick'),
    ])
    metric_label = {
        'mean_max_cos_sim':    'Mean max cosine similarity with true features',
        'frac_recovered_90':   'Fraction of features recovered (cos-sim ≥ 0.9)',
        'frac_recovered_80':   'Fraction of features recovered (cos-sim ≥ 0.8)',
    }.get(metric, metric)
    fig.update_layout(
        barmode='group',
        title=f'Feature recovery: SAE vs Temporal Crosscoder<br><sup>{metric_label}</sup>',
        yaxis_title=metric_label,
        yaxis_range=[0, 1.05],
        height=450, width=800,
        legend=dict(x=0.7, y=1),
    )
    fig.show()


# Grab true feature directions from toy model: (d, k)
TRUE_FEATURES = toy_model.embed.weight.detach()   # (d, num_feats)
print(f'True feature matrix shape: {TRUE_FEATURES.shape}')

# ── Update summary plot to show 3 models ─────────────────────────────────────
def plot_summary_bar(
    results: list[dict],
    metric: str = 'mean_max_cos_sim',
):
    labels     = [r['label'] for r in results]
    sae_shuf   = [r['sae_shuffled'][metric]   for r in results]
    sae_seq    = [r['sae_sequential'][metric] for r in results]
    cross_v    = [r['crosscoder'][metric]     for r in results]
    metric_label = {
        'mean_max_cos_sim':  'Mean max cosine similarity with true features',
        'frac_recovered_90': 'Fraction of features recovered (cos-sim ≥ 0.9)',
        'frac_recovered_80': 'Fraction of features recovered (cos-sim ≥ 0.8)',
    }.get(metric, metric)
    fig = go.Figure([
        go.Bar(name='SAE — shuffled (marginal)',    x=labels, y=sae_shuf, marker_color='steelblue'),
        go.Bar(name='SAE — sequential (in-order)',  x=labels, y=sae_seq,  marker_color='cornflowerblue'),
        go.Bar(name='Temporal Crosscoder',          x=labels, y=cross_v,  marker_color='firebrick'),
    ])
    fig.update_layout(
        barmode='group',
        title=f'Feature recovery: SAE (shuffled & sequential) vs Crosscoder<br><sup>{metric_label}</sup>',
        yaxis_title=metric_label, yaxis_range=[0, 1.05],
        height=480, width=900, legend=dict(x=0.55, y=1),
    )
    fig.show()

# Container for all results
ALL_RESULTS: list[dict] = []

True feature matrix shape: torch.Size([100, 50])


## 5b. Why the shuffled SAE always gets ~1.0 — covariance analysis

**The marginal argument.** All our temporal processes are *stationary*: the distribution at any single position $t$ is the same Bernoulli×|Normal| regardless of $t$ or of the inter-token correlation structure. When the SAE's `ShuffledDataIterator` picks a random timestep from each sequence, it sees exactly this marginal — an i.i.d. stream identical for IID, Scheme A, B, and C. So near-perfect recovery is expected and correct; it tells us nothing about temporal structure.

**The sequential question.** If we instead feed tokens in their natural temporal order (consecutive within a sequence), the SAE mini-batch has autocorrelated samples. The *effective* independent batch size shrinks by a factor related to the autocorrelation time:
$$
n_{\text{eff}} \approx \frac{n}{1 + 2\sum_{\tau=1}^{\infty} \rho_\tau} \approx \frac{n}{1 + 2\rho/(1-\rho)}
$$
For Scheme C with $\rho \approx 0.84$: $n_{\text{eff}} \approx n/12$. This should slow convergence and potentially degrade recovery — especially for rarely-fired features that only appear in short bursts.

**The covariance matrices below** show the $T \times T$ structure in each scheme and confirm that IID has diagonal structure while Schemes B and C have strong off-diagonal correlations.

In [31]:
# ── T×T Covariance & Correlation matrices per scheme ─────────────────────────

@torch.no_grad()
def compute_TxT_matrices(seq_gen_fn: Callable, N: int = 20_000) -> dict:
    """
    Compute empirical T×T covariance / correlation of the *scalar* activation
    a_{i,t} = s_{i,t} * m_{i,t}, averaged over all k features.
    Also compute tr(Cov(x_t, x_{t'})) — how that covariance propagates into R^d.
    """
    feat_seqs = seq_gen_fn(N, T_SEQ)                        # (N, T, k)
    act_seqs  = toy_model(feat_seqs.to(DEFAULT_DEVICE)).cpu()  # (N, T, d)
    feat_seqs = feat_seqs.cpu()

    k = feat_seqs.shape[-1]
    cov_sum  = torch.zeros(T_SEQ, T_SEQ)
    corr_sum = torch.zeros(T_SEQ, T_SEQ)

    for i in range(k):
        a   = feat_seqs[:, :, i]                            # (N, T)
        a_c = a - a.mean(dim=0)
        C   = (a_c.T @ a_c) / (N - 1)                      # (T, T) cov
        std = C.diag().sqrt().clamp(min=1e-8)
        R   = C / (std.unsqueeze(0) * std.unsqueeze(1))     # (T, T) corr
        cov_sum  += C
        corr_sum += R

    mean_cov  = cov_sum  / k
    mean_corr = corr_sum / k

    # tr(Cov(x_t, x_{t'})): sum of element-wise products over d dims
    act_c   = act_seqs - act_seqs.mean(dim=0)               # (N, T, d)
    rep_cov = torch.zeros(T_SEQ, T_SEQ)
    for t in range(T_SEQ):
        for tp in range(T_SEQ):
            rep_cov[t, tp] = (act_c[:, t, :] * act_c[:, tp, :]).sum(dim=-1).mean().item()

    return dict(cov=mean_cov, corr=mean_corr, rep_cov=rep_cov)


def print_TxT(scheme_name: str, seq_gen_fn: Callable, N: int = 20_000):
    mats  = compute_TxT_matrices(seq_gen_fn, N=N)
    W     = 72
    sep   = '  '

    def fmt_mat(M: torch.Tensor, fmt: str = '{:+.3f}') -> list[str]:
        return [sep.join(fmt.format(v) for v in row.tolist()) for row in M]

    print('\n' + '═' * W)
    print(f'  {scheme_name}')
    print('═' * W)

    print(f'\n  [1] Activation covariance  Cov(a_{{i,t}}, a_{{i,t\'}})  '
          f'(avg over {NUM_FEATS} features, N={N})')
    print(f'  Diagonal = Var(a_i,t).  Off-diagonal > 0 → temporal co-activation.')
    for line in fmt_mat(mats['cov']): print('    ' + line)

    print(f'\n  [2] Activation correlation  Corr(a_{{i,t}}, a_{{i,t\'}})  '
          f'(avg over {NUM_FEATS} features)')
    print(f'  IID → identity.  Scheme C → geometric decay at rate α-β.')
    for line in fmt_mat(mats['corr']): print('    ' + line)

    print(f'\n  [3] Representation covariance  tr(Cov(x_t, x_t\'))')
    print(f'  How activation cov propagates into R^d via the toy model.')
    print(f'  Large off-diagonal = temporal signal exploitable by the crosscoder.')
    for line in fmt_mat(mats['rep_cov']): print('    ' + line)

    c01 = mats['corr'][0, 1].item()
    r01 = mats['rep_cov'][0, 1].item()
    n_eff_ratio = 1 / max(1e-6, 1 + 2 * c01 / max(1e-6, 1 - c01)) if c01 < 1 else 0
    print(f'\n  Lag-1 summary:')
    print(f'    Corr(a_t, a_{{t+1}})            = {c01:+.5f}')
    print(f'    tr(Cov(x_t, x_{{t+1}}))         = {r01:+.5f}')
    print(f'    Effective batch fraction n_eff/n ≈ {n_eff_ratio:.4f}  '
          f'(=1 for IID, <1 for correlated → fewer effective gradient steps)')


# Define seq_gen_fns for schemes before experiments define per-exp versions
_scheme_gen_fns = [
    ('IID',                    lambda N, T: generate_iid_sequences(N, T)),
    ('Scheme A (AR(1) ρ=0.9)', lambda N, T: generate_scheme_a_sequences(N, T, rho=0.9)),
    ('Scheme B (copula ρ=0.85)',lambda N, T: generate_scheme_b_sequences(N, T, rho=0.85)),
    ('Scheme C (Markov α=0.95)',lambda N, T: generate_scheme_c_sequences(N, T, alpha=0.95, beta=0.03)),
]

for name, gfn in _scheme_gen_fns:
    print_TxT(name, gfn, N=20_000)


════════════════════════════════════════════════════════════════════════
  IID
════════════════════════════════════════════════════════════════════════

  [1] Activation covariance  Cov(a_{i,t}, a_{i,t'})  (avg over 50 features, N=20000)
  Diagonal = Var(a_i,t).  Off-diagonal > 0 → temporal co-activation.
    +0.048  -0.000  -0.000  -0.000  -0.000  +0.000  -0.000  -0.000
    -0.000  +0.049  -0.000  -0.000  -0.000  -0.000  -0.000  -0.000
    -0.000  -0.000  +0.048  +0.000  -0.000  +0.000  +0.000  -0.000
    -0.000  -0.000  +0.000  +0.049  +0.000  -0.000  -0.000  +0.000
    -0.000  -0.000  -0.000  +0.000  +0.049  +0.000  +0.000  +0.000
    +0.000  -0.000  +0.000  -0.000  +0.000  +0.049  -0.000  -0.000
    -0.000  -0.000  +0.000  -0.000  +0.000  -0.000  +0.048  +0.000
    -0.000  -0.000  -0.000  +0.000  +0.000  -0.000  +0.000  +0.049

  [2] Activation correlation  Corr(a_{i,t}, a_{i,t'})  (avg over 50 features)
  IID → identity.  Scheme C → geometric decay at rate α-β.
    +1.000  -0.000

---
## Experiment 1 — IID baseline (no temporal correlation)

Both SAE modes and the crosscoder see the same statistical signal; no temporal structure exists.
- **Shuffled SAE**: baseline, should be ~1.0.
- **Sequential SAE**: identical to shuffled for IID (no correlation to disrupt batch diversity).
- **Crosscoder**: should match or slightly lag SAE (extra encoder complexity with no signal to reward it).

> **Data**: $s_{i,t} \sim \text{Bernoulli}(0.05)$, $m_{i,t} \sim |\mathcal{N}(1.0, 0.15^2)|$, all i.i.d.

In [32]:
def iid_seq_gen(n: int, T: int) -> torch.Tensor:
    return generate_iid_sequences(n, T)

In [33]:
print('=== Exp 1: IID — SAE (shuffled) ===')
sae_exp1_shuf = train_standard_sae(iid_seq_gen, toy_model, mode='shuffled',
                                    d_sae=NUM_FEATS, training_tokens=TRAINING_TOKENS)
print('=== Exp 1: IID — SAE (sequential) ===')
sae_exp1_seq  = train_standard_sae(iid_seq_gen, toy_model, mode='sequential',
                                    d_sae=NUM_FEATS, training_tokens=TRAINING_TOKENS)

=== Exp 1: IID — SAE (shuffled) ===


Training SAE:   0%|          | 0/5000000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
print('=== Exp 1: IID — Temporal Crosscoder (TopK k={}) ==='.format(TOP_K))
# TopK: no l1_coeff needed. n_steps=CC_STEPS (defaults)
cc_exp1 = train_crosscoder(
    seq_gen_fn=iid_seq_gen, toy_model=toy_model,
    T=T_SEQ, d_sae=NUM_FEATS,
)


=== Exp 1: IID — Temporal Crosscoder (TopK k=11) ===
  Crosscoder (TopK k=11): 39,056 steps × 512 batch × T=8 = 159,973,376 token-equivalents  (SAE: 4,882 steps × 1024 = 5,000,000 tokens)


Training crosscoder:   0%|          | 0/39056 [00:00<?, ?it/s]

In [ ]:
m_sae1_shuf = feature_recovery_score(sae_exp1_shuf.W_dec.T, TRUE_FEATURES)
m_sae1_seq  = feature_recovery_score(sae_exp1_seq.W_dec.T,  TRUE_FEATURES)
m_cross1    = feature_recovery_score(cc_exp1.decoder_directions(pos=0), TRUE_FEATURES)

print('\n=== Exp 1 results ===')
print_recovery('SAE shuffled  ', m_sae1_shuf)
print_recovery('SAE sequential', m_sae1_seq)
print_recovery('Crosscoder    ', m_cross1)

ALL_RESULTS.append({
    'label': 'Exp 1\nIID',
    'sae_shuffled': m_sae1_shuf,
    'sae_sequential': m_sae1_seq,
    'crosscoder': m_cross1,
})

plot_recovery_heatmap(sae_exp1_shuf.W_dec.T, TRUE_FEATURES,
                      title='Exp 1 (IID) — SAE shuffled decoder vs true features')
plot_recovery_heatmap(sae_exp1_seq.W_dec.T, TRUE_FEATURES,
                      title='Exp 1 (IID) — SAE sequential decoder vs true features')
plot_recovery_heatmap(cc_exp1.decoder_directions(pos=0), TRUE_FEATURES,
                      title='Exp 1 (IID) — Crosscoder decoder (pos=0) vs true features')


=== Exp 1 results ===
[SAE shuffled  ]  mean-max cos-sim=1.000  | recovered@0.9=100.0%  | recovered@0.8=100.0%
[SAE sequential]  mean-max cos-sim=1.000  | recovered@0.9=100.0%  | recovered@0.8=100.0%
[Crosscoder    ]  mean-max cos-sim=0.455  | recovered@0.9=24.0%  | recovered@0.8=28.0%


---
## Experiment 2 — Scheme A: AR(1) magnitude correlation (ρ=0.9)

Magnitudes are temporally smooth but support is still i.i.d. Bernoulli.
As derived in the theory, activation autocorr ≈ γ·ρ ≈ p·ρ = 0.045 — very weak.
- **Shuffled SAE**: ≈ 1.0 (marginal unchanged).
- **Sequential SAE**: ≈ identical (activation autocorr too weak to affect batch diversity).
- **Crosscoder**: slight potential gain.


In [ ]:
RHO_A = 0.9

def seq_exp2_gen(n: int, T: int) -> torch.Tensor:
    return generate_scheme_a_sequences(n, T, rho=RHO_A)


In [ ]:
print('=== Exp 2: Scheme A (ρ=0.9) — SAE (shuffled) ===')
sae_exp2_shuf = train_standard_sae(seq_exp2_gen, toy_model, mode='shuffled',
                                         d_sae=NUM_FEATS, training_tokens=TRAINING_TOKENS)
print('=== Exp 2: Scheme A (ρ=0.9) — SAE (sequential) ===')
sae_exp2_seq  = train_standard_sae(seq_exp2_gen, toy_model, mode='sequential',
                                         d_sae=NUM_FEATS, training_tokens=TRAINING_TOKENS)


=== Exp 2: Scheme A (ρ=0.9) — SAE (shuffled) ===


Training SAE:   0%|          | 0/5000000 [00:00<?, ?it/s]

=== Exp 2: Scheme A (ρ=0.9) — SAE (sequential) ===


Training SAE:   0%|          | 0/5000000 [00:00<?, ?it/s]

In [ ]:
print('=== Exp 2: Scheme A (ρ=0.9) — Temporal Crosscoder (TopK k={}) ==='.format(TOP_K))
# TopK: no l1_coeff needed. n_steps=CC_STEPS (defaults)
cc_exp2 = train_crosscoder(
    seq_gen_fn=seq_exp2_gen, toy_model=toy_model,
    T=T_SEQ, d_sae=NUM_FEATS,
)


=== Exp 2: Scheme A (ρ=0.9) — Temporal Crosscoder (TopK k=11) ===
  Crosscoder (TopK k=11): 39,056 steps × 512 batch × T=8 = 159,973,376 token-equivalents  (SAE: 4,882 steps × 1024 = 5,000,000 tokens)


Training crosscoder:   0%|          | 0/39056 [00:00<?, ?it/s]

In [ ]:
m_sae2_shuf = feature_recovery_score(sae_exp2_shuf.W_dec.T, TRUE_FEATURES)
m_sae2_seq  = feature_recovery_score(sae_exp2_seq.W_dec.T,  TRUE_FEATURES)
m_cross2    = feature_recovery_score(cc_exp2.decoder_directions(pos=0), TRUE_FEATURES)

print(f'\n=== Exp 2 results ===')
print_recovery('SAE shuffled  ', m_sae2_shuf)
print_recovery('SAE sequential', m_sae2_seq)
print_recovery('Crosscoder    ', m_cross2)

ALL_RESULTS.append({
    'label': 'Exp 2\nScheme A (ρ=0.9)',
    'sae_shuffled':   m_sae2_shuf,
    'sae_sequential': m_sae2_seq,
    'crosscoder':     m_cross2,
})

plot_recovery_heatmap(sae_exp2_shuf.W_dec.T, TRUE_FEATURES,
                      title='Exp 2 (Scheme A (ρ=0.9)) — SAE shuffled')
plot_recovery_heatmap(sae_exp2_seq.W_dec.T,  TRUE_FEATURES,
                      title='Exp 2 (Scheme A (ρ=0.9)) — SAE sequential')
plot_recovery_heatmap(cc_exp2.decoder_directions(pos=0), TRUE_FEATURES,
                      title='Exp 2 (Scheme A (ρ=0.9)) — Crosscoder (pos=0)')



=== Exp 2 results ===
[SAE shuffled  ]  mean-max cos-sim=1.000  | recovered@0.9=100.0%  | recovered@0.8=100.0%
[SAE sequential]  mean-max cos-sim=1.000  | recovered@0.9=100.0%  | recovered@0.8=100.0%
[Crosscoder    ]  mean-max cos-sim=0.479  | recovered@0.9=36.0%  | recovered@0.8=36.0%


---
## Experiment 3 — Scheme B: Gaussian copula support (ρ=0.85)

Support process is correlated; activation lag-1 autocorr ≈ 0.52 (see covariance cell).
- **Shuffled SAE**: ≈ 1.0.
- **Sequential SAE**: effective batch shrinks by ~1/(1+2·0.52/(1-0.52)) ≈ 0.31 → may degrade slightly.
- **Crosscoder**: moderate expected gain.


In [ ]:
RHO_B = 0.85

def seq_exp3_gen(n: int, T: int) -> torch.Tensor:
    return generate_scheme_b_sequences(n, T, rho=RHO_B)


In [ ]:
print('=== Exp 3: Scheme B (ρ=0.85) — SAE (shuffled) ===')
sae_exp3_shuf = train_standard_sae(seq_exp3_gen, toy_model, mode='shuffled',
                                         d_sae=NUM_FEATS, training_tokens=TRAINING_TOKENS)
print('=== Exp 3: Scheme B (ρ=0.85) — SAE (sequential) ===')
sae_exp3_seq  = train_standard_sae(seq_exp3_gen, toy_model, mode='sequential',
                                         d_sae=NUM_FEATS, training_tokens=TRAINING_TOKENS)


=== Exp 3: Scheme B (ρ=0.85) — SAE (shuffled) ===


Training SAE:   0%|          | 0/5000000 [00:00<?, ?it/s]

=== Exp 3: Scheme B (ρ=0.85) — SAE (sequential) ===


Training SAE:   0%|          | 0/5000000 [00:00<?, ?it/s]

In [ ]:
print('=== Exp 3: Scheme B (ρ=0.85) — Temporal Crosscoder (TopK k={}) ==='.format(TOP_K))
# TopK: no l1_coeff needed. n_steps=CC_STEPS (defaults)
cc_exp3 = train_crosscoder(
    seq_gen_fn=seq_exp3_gen, toy_model=toy_model,
    T=T_SEQ, d_sae=NUM_FEATS,
)


=== Exp 3: Scheme B (ρ=0.85) — Temporal Crosscoder (TopK k=11) ===
  Crosscoder (TopK k=11): 39,056 steps × 512 batch × T=8 = 159,973,376 token-equivalents  (SAE: 4,882 steps × 1024 = 5,000,000 tokens)


Training crosscoder:   0%|          | 0/39056 [00:00<?, ?it/s]

In [ ]:
m_sae3_shuf = feature_recovery_score(sae_exp3_shuf.W_dec.T, TRUE_FEATURES)
m_sae3_seq  = feature_recovery_score(sae_exp3_seq.W_dec.T,  TRUE_FEATURES)
m_cross3    = feature_recovery_score(cc_exp3.decoder_directions(pos=0), TRUE_FEATURES)

print(f'\n=== Exp 3 results ===')
print_recovery('SAE shuffled  ', m_sae3_shuf)
print_recovery('SAE sequential', m_sae3_seq)
print_recovery('Crosscoder    ', m_cross3)

ALL_RESULTS.append({
    'label': 'Exp 3\nScheme B (ρ=0.85)',
    'sae_shuffled':   m_sae3_shuf,
    'sae_sequential': m_sae3_seq,
    'crosscoder':     m_cross3,
})

plot_recovery_heatmap(sae_exp3_shuf.W_dec.T, TRUE_FEATURES,
                      title='Exp 3 (Scheme B (ρ=0.85)) — SAE shuffled')
plot_recovery_heatmap(sae_exp3_seq.W_dec.T,  TRUE_FEATURES,
                      title='Exp 3 (Scheme B (ρ=0.85)) — SAE sequential')
plot_recovery_heatmap(cc_exp3.decoder_directions(pos=0), TRUE_FEATURES,
                      title='Exp 3 (Scheme B (ρ=0.85)) — Crosscoder (pos=0)')



=== Exp 3 results ===
[SAE shuffled  ]  mean-max cos-sim=1.000  | recovered@0.9=100.0%  | recovered@0.8=100.0%
[SAE sequential]  mean-max cos-sim=1.000  | recovered@0.9=100.0%  | recovered@0.8=100.0%
[Crosscoder    ]  mean-max cos-sim=0.927  | recovered@0.9=72.0%  | recovered@0.8=92.0%


---
## Experiment 4 — Scheme C: 2-state Markov chain (α=0.95, β=0.03)

Strongest temporal correlation. Lag-1 autocorr ≈ 0.88. Features stay active for long bursts.
- **Shuffled SAE**: ≈ 1.0.
- **Sequential SAE**: effective batch ≈ n/15 → expected meaningful degradation for rare features.
- **Crosscoder**: largest expected gain — sustained activations provide maximal joint evidence.


In [34]:
ALPHA_C = 0.95
BETA_C  = 0.03
p_stat_c = BETA_C / (1 - ALPHA_C + BETA_C)
print(f'Scheme C: stationary p={p_stat_c:.3f}  lag-1 autocorr={ALPHA_C-BETA_C:.3f}')

def seq_exp4_gen(n: int, T: int) -> torch.Tensor:
    return generate_scheme_c_sequences(n, T, alpha=ALPHA_C, beta=BETA_C)


Scheme C: stationary p=0.375  lag-1 autocorr=0.920


In [35]:
print('=== Exp 4: Scheme C (α=0.95) — SAE (shuffled) ===')
sae_exp4_shuf = train_standard_sae(seq_exp4_gen, toy_model, mode='shuffled',
                                         d_sae=NUM_FEATS, training_tokens=TRAINING_TOKENS)
print('=== Exp 4: Scheme C (α=0.95) — SAE (sequential) ===')
sae_exp4_seq  = train_standard_sae(seq_exp4_gen, toy_model, mode='sequential',
                                         d_sae=NUM_FEATS, training_tokens=TRAINING_TOKENS)


=== Exp 4: Scheme C (α=0.95) — SAE (shuffled) ===


Training SAE:   0%|          | 0/5000000 [00:00<?, ?it/s]

=== Exp 4: Scheme C (α=0.95) — SAE (sequential) ===


Training SAE:   0%|          | 0/5000000 [00:00<?, ?it/s]

In [36]:
print('=== Exp 4: Scheme C (α=0.95) — Temporal Crosscoder (TopK k={}) ==='.format(TOP_K))
# TopK: no l1_coeff needed. n_steps=CC_STEPS (defaults)
cc_exp4 = train_crosscoder(
    seq_gen_fn=seq_exp4_gen, toy_model=toy_model,
    T=T_SEQ, d_sae=NUM_FEATS,
)


=== Exp 4: Scheme C (α=0.95) — Temporal Crosscoder (TopK k=17) ===
  Crosscoder (TopK k=17): 39,056 steps × 512 batch × T=8 = 159,973,376 token-equivalents  (SAE: 4,882 steps × 1024 = 5,000,000 tokens)


Training crosscoder:   0%|          | 0/39056 [00:00<?, ?it/s]

In [37]:
m_sae4_shuf = feature_recovery_score(sae_exp4_shuf.W_dec.T, TRUE_FEATURES)
m_sae4_seq  = feature_recovery_score(sae_exp4_seq.W_dec.T,  TRUE_FEATURES)
m_cross4    = feature_recovery_score(cc_exp4.decoder_directions(pos=0), TRUE_FEATURES)

print(f'\n=== Exp 4 results ===')
print_recovery('SAE shuffled  ', m_sae4_shuf)
print_recovery('SAE sequential', m_sae4_seq)
print_recovery('Crosscoder    ', m_cross4)

ALL_RESULTS.append({
    'label': 'Exp 4\nScheme C (α=0.95)',
    'sae_shuffled':   m_sae4_shuf,
    'sae_sequential': m_sae4_seq,
    'crosscoder':     m_cross4,
})

plot_recovery_heatmap(sae_exp4_shuf.W_dec.T, TRUE_FEATURES,
                      title='Exp 4 (Scheme C (α=0.95)) — SAE shuffled')
plot_recovery_heatmap(sae_exp4_seq.W_dec.T,  TRUE_FEATURES,
                      title='Exp 4 (Scheme C (α=0.95)) — SAE sequential')
plot_recovery_heatmap(cc_exp4.decoder_directions(pos=0), TRUE_FEATURES,
                      title='Exp 4 (Scheme C (α=0.95)) — Crosscoder (pos=0)')



=== Exp 4 results ===
[SAE shuffled  ]  mean-max cos-sim=0.877  | recovered@0.9=50.0%  | recovered@0.8=76.0%
[SAE sequential]  mean-max cos-sim=0.765  | recovered@0.9=24.0%  | recovered@0.8=46.0%
[Crosscoder    ]  mean-max cos-sim=0.632  | recovered@0.9=50.0%  | recovered@0.8=50.0%


---
## Summary — all experiments

We compare feature recovery across all four conditions using three metrics:
1. **Mean max cosine similarity** — averaged over true features.
2. **Recovery @ 0.9** — fraction of true features with a crosscoder latent aligned to ≥ 0.9 cosine similarity.
3. **Recovery @ 0.8** — same with a looser threshold.

In [38]:
# ── Summary table ──────────────────────────────────────────────────────────
xc_tokens = SAE_STEPS * CC_BATCH * T_SEQ
print(f'Token exposure — SAE: {TRAINING_TOKENS:,}  |  Crosscoder: {xc_tokens:,} ({xc_tokens/TRAINING_TOKENS:.1f}×)')
print(f'Gradient steps — SAE: {SAE_STEPS:,}  |  Crosscoder: {SAE_STEPS:,} (matched)\n')

print(f'{"Exp":<22} {"SAE-shuf @0.9":>14} {"SAE-seq @0.9":>13} {"XC @0.9":>9}'
      f' {"SAE-shuf mmcs":>14} {"SAE-seq mmcs":>13} {"XC mmcs":>9}')
print('-' * 100)
for r in ALL_RESULTS:
    s, q, x = r['sae_shuffled'], r['sae_sequential'], r['crosscoder']
    label = r['label'].replace('\n', ' ')
    xc_gain = x['mean_max_cos_sim'] - max(s['mean_max_cos_sim'], q['mean_max_cos_sim'])
    flag = '  ← XC wins' if xc_gain > 0.02 else ('  ← XC loses' if xc_gain < -0.05 else '')
    print(f'{label:<22} {s["frac_recovered_90"]:>14.1%} {q["frac_recovered_90"]:>13.1%} {x["frac_recovered_90"]:>9.1%}'
          f' {s["mean_max_cos_sim"]:>14.3f} {q["mean_max_cos_sim"]:>13.3f} {x["mean_max_cos_sim"]:>9.3f}{flag}')

for metric in ['mean_max_cos_sim', 'frac_recovered_90']:
    plot_summary_bar(ALL_RESULTS, metric=metric)


Token exposure — SAE: 5,000,000  |  Crosscoder: 19,996,672 (4.0×)
Gradient steps — SAE: 4,882  |  Crosscoder: 4,882 (matched)

Exp                     SAE-shuf @0.9  SAE-seq @0.9   XC @0.9  SAE-shuf mmcs  SAE-seq mmcs   XC mmcs
----------------------------------------------------------------------------------------------------
Exp 4 Scheme C (α=0.95)          50.0%         24.0%     50.0%          0.877         0.765     0.632  ← XC loses


In [40]:
import numpy as np

# ── AUC over cosine similarity thresholds ─────────────────────────────────────

THRESHOLDS = np.linspace(0, 1, 200)

def recovery_curve(per_feature: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    return np.array([(per_feature >= t).mean() for t in thresholds])

def auc(curve: np.ndarray, thresholds: np.ndarray) -> float:
    return float(np.trapz(curve, thresholds) / (thresholds[-1] - thresholds[0]))


# ── Compute curves and AUCs ───────────────────────────────────────────────────
print(f'{"Exp":<22} {"SAE-shuf AUC":>13} {"SAE-seq AUC":>12} {"XC AUC":>9} {"XC vs shuf":>11}')
print('-' * 72)

auc_results = []
for r in ALL_RESULTS:
    label = r['label'].replace('\n', ' ')
    curves = {}
    aucs   = {}
    for key in ('sae_shuffled', 'sae_sequential', 'crosscoder'):
        curves[key] = recovery_curve(r[key]['per_feature'], THRESHOLDS)
        aucs[key]   = auc(curves[key], THRESHOLDS)

    delta = aucs['crosscoder'] - aucs['sae_shuffled']
    flag  = '  <- XC wins' if delta > 0.01 else ('  <- XC loses' if delta < -0.01 else '  ~')
    print(f'{label:<22} {aucs["sae_shuffled"]:>13.4f} {aucs["sae_sequential"]:>12.4f} '
          f'{aucs["crosscoder"]:>9.4f} {delta:>+11.4f}{flag}')

    auc_results.append({'label': r['label'], 'curves': curves, 'aucs': aucs})


# ── Plot recovery curves ──────────────────────────────────────────────────────
model_styles = {
    'sae_shuffled':   ('SAE - shuffled',    'steelblue',      'solid'),
    'sae_sequential': ('SAE - sequential',  'cornflowerblue', 'dot'),
    'crosscoder':     ('Crosscoder (TopK)', 'firebrick',      'solid'),
}

fig = make_subplots(
    rows=1, cols=len(auc_results),
    subplot_titles=[r['label'].replace('\n', '<br>') for r in auc_results],
    shared_yaxes=True,
)

for col, r in enumerate(auc_results, start=1):
    for key, (name, color, dash) in model_styles.items():
        fig.add_trace(
            go.Scatter(
                x=THRESHOLDS,
                y=r['curves'][key],
                name=name,
                line=dict(color=color, dash=dash, width=2),
                showlegend=(col == 1),
                legendgroup=key,
                hovertemplate=f'{name}<br>threshold=%{{x:.2f}}<br>frac=%{{y:.3f}}<extra></extra>',
            ),
            row=1, col=col,
        )

    # AUC annotations using paper coords (avoids x1 domain / xref validation bug)
    n_cols   = len(auc_results)
    pw       = 1.0 / n_cols
    x_center = (col - 1) * pw + pw * 0.62
    for i, (key, (name, color, _)) in enumerate(model_styles.items()):
        fig.add_annotation(
            x=x_center, y=0.95 - i * 0.12,
            xref='paper', yref='paper',
            text=f'AUC={r["aucs"][key]:.3f}',
            font=dict(color=color, size=10),
            showarrow=False,
        )

fig.update_layout(
    title='Feature recovery curves - fraction recovered vs cosine similarity threshold<br>'
          '<sup>AUC summarises recovery across all thresholds simultaneously</sup>',
    height=420, width=1100,
    legend=dict(x=0.01, y=-0.2, orientation='h'),
)
fig.update_xaxes(title_text='cos-sim threshold', range=[0, 1])
fig.update_yaxes(title_text='Fraction recovered', range=[0, 1.05], col=1)
fig.show()


# ── AUC bar chart ─────────────────────────────────────────────────────────────
labels = [r['label'].replace('\n', ' ') for r in auc_results]
fig2 = go.Figure([
    go.Bar(name='SAE - shuffled',    x=labels,
           y=[r['aucs']['sae_shuffled']   for r in auc_results], marker_color='steelblue'),
    go.Bar(name='SAE - sequential',  x=labels,
           y=[r['aucs']['sae_sequential'] for r in auc_results], marker_color='cornflowerblue'),
    go.Bar(name='Crosscoder (TopK)', x=labels,
           y=[r['aucs']['crosscoder']     for r in auc_results], marker_color='firebrick'),
])
fig2.update_layout(
    barmode='group',
    title='AUC of feature recovery curve (higher = better across all thresholds)',
    yaxis=dict(title='AUC', range=[0, 1.05]),
    height=420, width=800,
    legend=dict(x=0.6, y=1),
)
fig2.show()

Exp                     SAE-shuf AUC  SAE-seq AUC    XC AUC  XC vs shuf
------------------------------------------------------------------------
Exp 4 Scheme C (α=0.95)        0.8776       0.7649    0.6325     -0.2451  <- XC loses


---
## Discussion

### What we expect to see

| Experiment | SAE | Crosscoder | Reason |
|-----------|-----|------------|--------|
| IID | baseline | ≈ SAE | No temporal signal exists |
| Scheme A (AR(1) mags) | baseline | slight gain | Weak signal due to sparsity attenuation $\gamma \leq p = 0.05$ |
| Scheme B (copula support) | baseline | moderate gain | Correlated support gives crosscoder more co-occurrence evidence |
| Scheme C (Markov support) | baseline | largest gain | Strong, sustained activations make joint encoding much easier |

### Intuition for _why_ the crosscoder helps

A standard SAE sees each position independently. Under IID data, this is no loss of information. But when the support process is correlated, observing $x_t$ gives information about $x_{t+1}$: if feature $i$ is active now, it's likely still active next step. The crosscoder's shared encoder sees the whole window $[x_t, \ldots, x_{t+T-1}]$ simultaneously and can use this joint evidence to more confidently identify which features are active, effectively "denoising" through temporal pooling.

### Limitations of this toy model

- We use **ReLU** (not TopK or JumpReLU) for simplicity; a TopK crosscoder would enforce exact sparsity and may perform differently.
- The crosscoder training loop is custom (not via sae-lens) and may not be perfectly comparable in terms of hyperparameter tuning.
- Real LLM activations have many more features and more complex temporal structure — but this toy setup establishes the principle.

### Next steps

- **Sweep $T$**: does a longer window continue to help, or does the marginal benefit saturate?
- **Sweep $\rho / \alpha$**: map out the phase transition where the crosscoder begins outperforming the SAE.
- **Superposition**: apply a bottleneck $\mathbf{W}_\text{model} \in \mathbb{R}^{d' \times d}$ with $d' < k$ and see whether temporal information also aids feature recovery under superposition.
- **Real models**: apply temporal crosscoders to GPT-2 or Llama activations and compare feature interpretability scores.